<table width="100%" bgcolor="#f4f5f6" style="width: 100%; background-color: #f4f5f6; border-left: 6px solid #ff9900; border-radius: 8px; border-collapse: collapse;">
  <tr>
    <td width="110" align="center" valign="middle" style="padding: 15px;">
      <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block;">
    </td>
    <td valign="middle" style="padding: 15px 20px 15px 5px; font-family: Arial, sans-serif;">
      <p style="margin: 0 0 4px 0; font-size: 13px; color: #555555;">Excerpt from the Amazon book</p>
      <h2 style="margin: 4px 0 10px 0; font-size: 20px; font-weight: 900; color: #111111; border: none; padding: 0;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</h2>
      <p style="margin: 0; font-size: 14px; font-weight: bold;">
        <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: underline;">🛒 Buy on Amazon &rarr;</a>
      </p>
    </td>
  </tr>
</table>



## **Delta Lake (Ideal for transactional databases, advanced analytics, and version control)**
Delta Lake adds ACID transactions on top of Parquet files.

In [1]:
# Install compatible versions of PySpark and Delta Lake to avoid conflicts
# Ignore the warning that this cell will trigger and proceed.
!pip install -q pyspark==3.5.4 delta-spark==3.2.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.4 which is incompatible.


In [2]:
# clone Github repository
!rm -rf shining-path_DP700
!git clone -q https://github.com/sansalamanca/shining-path_DP700.git


In [ ]:
# Mount your Google Drive to work with Colab
from google.colab import drive
drive.mount('/content/drive')

In [3]:
# Initialize PySpark with Delta Lake support

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Configure the builder with the extensions and the Delta catalog
builder = SparkSession.builder \
    .appName("Colab_Session_10") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Initialize the session by automatically downloading the correct Delta Lake package
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# Define the schema
schema = StructType() \
    .add("ProductID", IntegerType(), True) \
    .add("ProductName", StringType(), True) \
    .add("Category", StringType(), True) \
    .add("ListPrice", DoubleType(), True)


In [5]:
# Read the data from the repository and display the DF
df = spark.read.format("csv").option("header", "True").load("shining-path_DP700/data/products.csv")
df.show()  # In Fabric use display(df)

+---------+--------------------+--------------+---------+
|ProductID|         ProductName|      Category|ListPrice|
+---------+--------------------+--------------+---------+
|      771|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      772|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      773|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      774|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      775|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      776|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      777|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      778|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      779|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      780|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      781|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      782|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      783|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      784|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      785|Mou

In [10]:
# 1. Read the data, applying the previously defined schema to ensure the correct data types

df_products = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("shining-path_DP700/data/products.csv")

# 2. Save the DataFrame directly as a Delta table in your Google Drive
df_products.write.format("delta").mode("overwrite").save(
    "/content/drive/MyDrive/my_tables_deltalake/products_delta"
)

print("The Delta table has been successfully saved to your Google Drive using the active Delta-enabled session.")

The Delta table has been successfully saved to your Google Drive using the active Delta-enabled session.


In [13]:
# 1. Read the Delta table saved in Google Drive
path_delta = "/content/drive/MyDrive/my_tables_deltalake/products_delta"
outcome_df = spark.read.format("delta").load(path_delta)

# 2. Display the content and its schema for verification
outcome_df.show(10, truncate=False)
outcome_df.printSchema()

+---------+-----------------------+--------------+---------+
|ProductID|ProductName            |Category      |ListPrice|
+---------+-----------------------+--------------+---------+
|771      |Mountain-100 Silver, 38|Mountain Bikes|3399.99  |
|772      |Mountain-100 Silver, 42|Mountain Bikes|3399.99  |
|773      |Mountain-100 Silver, 44|Mountain Bikes|3399.99  |
|774      |Mountain-100 Silver, 48|Mountain Bikes|3399.99  |
|775      |Mountain-100 Black, 38 |Mountain Bikes|3374.99  |
|776      |Mountain-100 Black, 42 |Mountain Bikes|3374.99  |
|777      |Mountain-100 Black, 44 |Mountain Bikes|3374.99  |
|778      |Mountain-100 Black, 48 |Mountain Bikes|3374.99  |
|779      |Mountain-200 Silver, 38|Mountain Bikes|2319.99  |
|780      |Mountain-200 Silver, 42|Mountain Bikes|2319.99  |
+---------+-----------------------+--------------+---------+
only showing top 10 rows

root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullabl